# IR System — LTR for MSMARCO
**Step 11:** Train the Learning-to-Rank model on the MSMARCO dataset and save it.

Just run all cells in order (Runtime → Run all).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/ir_system_data'
import os, sys

if not os.path.exists('/content/ir-system'):
    !git clone https://github.com/ghazal-mohammad/ir-system.git /content/ir-system
else:
    !cd /content/ir-system && git pull
sys.path.insert(0, '/content/ir-system')

!pip install scikit-learn -q
print('ready')

In [ ]:
# Load saved MSMARCO retrieval results + qrels
import json

def load_res(path):
    with open(path) as f: return json.load(f)

results_per_model = {
    'bm25':      load_res(f'{SAVE_DIR}/msmarco_bm25_results.json'),
    'tfidf':     load_res(f'{SAVE_DIR}/msmarco_tfidf_results.json'),
    'embedding': load_res(f'{SAVE_DIR}/msmarco_embedding_results.json'),
}

qrels_raw = load_res(f'{SAVE_DIR}/msmarco_qrels.json')
qrels = {qid: {d: r for d, r in docs.items() if r >= 1}
         for qid, docs in qrels_raw.items()}
qrels = {qid: docs for qid, docs in qrels.items() if docs}

query_ids = list(qrels.keys())
print(f'Queries with qrels: {len(query_ids)}')

In [ ]:
# Train/test split + train the LTR model
from services.ltr_service import (
    build_feature_matrix_multi, train_ltr_model,
    rerank_with_ltr, evaluate_ltr, save_ltr_model
)

split = int(len(query_ids) * 0.7)
train_qids = query_ids[:split]
test_qids  = query_ids[split:]

model_names = list(results_per_model.keys())

X_train, y_train, _ = build_feature_matrix_multi(train_qids, results_per_model, qrels)
X_test,  y_test,  _ = build_feature_matrix_multi(test_qids,  results_per_model, qrels)
print(f'Train: {X_train.shape}, positives: {y_train.sum()}')
print(f'Test:  {X_test.shape},  positives: {y_test.sum()}')

ltr = train_ltr_model(X_train, y_train)
print('Train:', evaluate_ltr(X_train, y_train, ltr))
print('Test: ', evaluate_ltr(X_test,  y_test,  ltr))

save_ltr_model(ltr, f'{SAVE_DIR}/msmarco_ltr_model.pkl')
print('Saved: msmarco_ltr_model.pkl')

In [ ]:
# Compare LTR vs baselines on the test queries
from services.evaluation_service import evaluate_run, print_results_table

ltr_results = {}
for qid in test_qids:
    ltr_results[qid] = rerank_with_ltr(qid, results_per_model, ltr, model_names, top_k=1000)

ltr_eval  = evaluate_run(ltr_results, qrels)['aggregated']
bm25_eval = evaluate_run({q: results_per_model['bm25'][q] for q in test_qids
                          if q in results_per_model['bm25']}, qrels)['aggregated']
emb_eval  = evaluate_run({q: results_per_model['embedding'][q] for q in test_qids
                          if q in results_per_model['embedding']}, qrels)['aggregated']

print('=== LTR vs Baselines (MSMARCO test set) ===')
print_results_table({'BM25': bm25_eval, 'Embedding': emb_eval, 'LTR': ltr_eval})

with open(f'{SAVE_DIR}/msmarco_ltr_results.json', 'w') as f:
    json.dump(ltr_results, f)
print('\nDone — msmarco_ltr_results.json saved')